In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Parametri comunemente usati per la transpilazione

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Consigliamo di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Questa pagina descrive alcuni dei parametri più comunemente usati per la transpilazione locale. Questi parametri vengono configurati tramite argomenti di [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) o [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Grado di approssimazione
Puoi usare il grado di approssimazione per specificare quanto vuoi che il circuito risultante corrisponda al circuito desiderato (di input). Si tratta di un valore float nell'intervallo (0.0 - 1.0), dove 0.0 indica la massima approssimazione e 1.0 (valore predefinito) indica nessuna approssimazione. Valori più bassi scambiano l'accuratezza dell'output con una maggiore facilità di esecuzione (cioè, meno gate). Il valore predefinito è 1.0.

Nella sintesi unitaria a due qubit (usata nelle fasi iniziali di tutti i livelli e nella fase di ottimizzazione con livello di ottimizzazione 3), questo valore specifica la fedeltà target della decomposizione in output. In altre parole, quanti errori vengono introdotti quando una rappresentazione matriciale di un circuito viene convertita in gate discreti. Se il grado di approssimazione è più basso (maggiore approssimazione), il circuito in output dalla sintesi si discosterà di più dalla matrice di input, ma probabilmente avrà anche meno gate (poiché qualsiasi operazione arbitraria a due qubit può essere decomposta perfettamente con al massimo tre gate CX) e sarà più facile da eseguire.

Quando il grado di approssimazione è inferiore a 1.0, potrebbero essere sintetizzati circuiti con uno o due gate CX, portando a meno errori dall'hardware ma più errori dall'approssimazione. Poiché CX è il gate più costoso in termini di errori, potrebbe essere vantaggioso ridurne il numero a scapito della fedeltà nella sintesi (questa tecnica è stata usata per aumentare il quantum volume sui dispositivi IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Come esempio, generiamo un `UnitaryGate` casuale a due qubit che verrà sintetizzato nella fase iniziale. Impostare `approximation_degree` a un valore inferiore a 1.0 potrebbe generare un circuito approssimato. Dobbiamo anche specificare i `basis_gates` per comunicare al metodo di sintesi quali gate può usare per la sintesi approssimata.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Il risultato è `2` perché l'approssimazione richiede meno gate CX.

<span id="seed"></span>
## Seed del generatore di numeri casuali
Alcune parti del transpiler sono stocastiche, quindi esecuzioni ripetute della transpilazione possono restituire risultati diversi. Per ottenere un risultato riproducibile, puoi impostare il seed del generatore di numeri pseudocasuali usando l'argomento `seed_transpiler`. Esecuzioni ripetute con lo stesso seed restituiranno gli stessi risultati.

Esempio:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Layout iniziale
Prima della transpilazione, i qubit contenuti nel tuo circuito sono qubit virtuali che non corrispondono necessariamente ai qubit fisici del backend target. Puoi specificare la mappatura iniziale dai qubit virtuali ai qubit fisici usando l'argomento `initial_layout`. Nota che il layout finale dei qubit potrebbe differire da quello iniziale, poiché il transpiler potrebbe permutare i qubit tramite swap gate o altri metodi.

Nell'esempio seguente, costruiamo un layout iniziale per il mock backend [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) creando un oggetto [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Il nostro layout mappa il primo qubit del circuito al qubit 5 di Sherbrooke e il secondo qubit del circuito al qubit 6 di Sherbrooke. Nota che i qubit fisici sono sempre rappresentati da interi.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Oltre a specificare un oggetto Layout, puoi anche passare una lista di interi, dove l'$i$-esimo elemento della lista contiene il qubit fisico a cui dovrebbe essere mappato l'$i$-esimo qubit. Per esempio:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Puoi usare la funzione [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) per generare un diagramma del grafo del dispositivo con informazioni sugli errori e con i qubit fisici etichettati. Puoi anche visualizzare diagrammi simili nella pagina [Compute resources](https://quantum.cloud.ibm.com/computers).